# Tutorial 1: Test-Time Training End-to-End (TTT-E2E)

A deep dive into the most sophisticated continual learning strategy in this project.

---

## Table of Contents

1. [Introduction](#1-introduction)
2. [Concepts: The DualMLP Architecture](#2-concepts-the-dualmlp-architecture)
3. [Concepts: TF-IDF Gating](#3-concepts-tf-idf-gating)
4. [Concepts: Alpha Decay](#4-concepts-alpha-decay)
5. [Hands-On: Loading and Modifying the Model](#5-hands-on-loading-and-modifying-the-model)
6. [Hands-On: Learning a Document](#6-hands-on-learning-a-document)
7. [Hands-On: Testing Knowledge](#7-hands-on-testing-knowledge)
8. [Hands-On: Checkpointing](#8-hands-on-checkpointing)
9. [Deep Dive: The Math](#9-deep-dive-the-math)
10. [Exercises](#10-exercises)

---

## 1. Introduction

### What is Test-Time Training?

Traditional language models follow a simple lifecycle:

1. **Pre-training** on massive corpora (billions of tokens)
2. **Fine-tuning** on task-specific data
3. **Inference** with frozen weights

Once deployed, the model is **frozen**. It cannot learn from the documents you feed it.
If you show it a new research paper, a company report, or a user manual, it will process
the text through its fixed weights, but those weights will not change. The knowledge
stays external -- it never becomes part of the model.

**Test-Time Training (TTT)** breaks this paradigm. The model keeps learning *during inference*.
When you present a new document, the model performs gradient descent on that document's tokens,
updating a subset of its weights so that the new information is *internalized*.

### Why Modify Weights at Inference?

You might ask: why not just use retrieval-augmented generation (RAG) or long context windows?

| Approach | Pros | Cons |
|----------|------|------|
| **Long context** | Simple | Expensive at inference, O(n^2) attention |
| **RAG** | Scalable retrieval | Lossy, retrieval errors, no deep understanding |
| **Fine-tuning** | Deep learning | Slow, expensive, catastrophic forgetting |
| **TTT-E2E** | Deep learning, fast, selective | More complex architecture |

TTT-E2E gives us the depth of fine-tuning with the speed and selectivity we need at inference time.

### The Core Challenge: Catastrophic Forgetting

The fundamental problem with updating weights at inference time is **catastrophic forgetting**.
If we naively fine-tune on a new document, the model loses its general language abilities.
It might learn the new facts, but forget how to form grammatical sentences.

TTT-E2E addresses this with three mechanisms:

1. **DualMLP Architecture** -- A frozen MLP preserves original knowledge; a trainable MLP learns new information
2. **TF-IDF Gating** -- Gradient masks prevent updates to general-purpose neurons
3. **Alpha Decay** -- A blending coefficient controls how much influence the trainable MLP has

---

## 2. Concepts: The DualMLP Architecture

### Background: MLPs in Transformers

Each transformer layer in Qwen2.5 has two main components:
- **Self-Attention** -- determines *which* tokens to focus on
- **MLP (Feed-Forward Network)** -- transforms token representations through learned projections

The MLP is where factual knowledge is primarily stored. Research has shown that
modifying MLP weights is the most direct way to update a model's knowledge.

### The DualMLP Idea

Instead of modifying the original MLP (which would destroy pre-trained knowledge),
we **add a second MLP alongside it**:

```
                        +------------------+
             +--------->| Frozen MLP       |--------+
             |          | (original weights)|       |
             |          +------------------+        |
   input x --+                                      +--> output
             |          +------------------+        |     = frozen_out + (1-alpha) * gated_trainable_out
             +--------->| Trainable MLP    |--+     |
                        | (learns new info)|  |     |
                        +------------------+  |     |
                                              v     |
                        +------------------+  |     |
                        | TF-IDF Gate      |--+-----+
                        | (gradient mask)  |  scaled by (1-alpha)
                        +------------------+
```

### The Formula

$$\text{output} = \text{frozen\_mlp}(x) + (1 - \alpha) \cdot \text{gate}(\text{trainable\_mlp}(x))$$

Where:
- **frozen_mlp(x)** -- the original, unmodified MLP output (always preserved)
- **trainable_mlp(x)** -- the new MLP that learns from documents
- **gate()** -- the TF-IDF mask that selects which neurons contribute
- **alpha** -- blending coefficient, starts at 0.95 (mostly frozen output)

When alpha = 1.0, the output is purely the frozen MLP (no new knowledge).
When alpha = 0.5, the frozen and trainable MLPs contribute equally.

### SwiGLU Activation

Both MLPs use the **SwiGLU** activation function, which is standard in modern transformers:

$$\text{SwiGLU}(x) = \text{down\_proj}\big(\text{SiLU}(\text{gate\_proj}(x)) \cdot \text{up\_proj}(x)\big)$$

This has three weight matrices:
- `gate_proj`: hidden_size -> intermediate_size
- `up_proj`: hidden_size -> intermediate_size
- `down_proj`: intermediate_size -> hidden_size

The trainable MLP is initialized with small random weights (std=0.001) rather than zeros.
This avoids the SwiGLU dead gradient problem: if gate_proj outputs are zero, the SiLU
gradient is also near zero, preventing any learning.

In [ ]:
import torch
import torch.nn as nn
import sys
import os

# Ensure the project source is importable
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, os.path.join(project_root, "src"))

from continual_learning.model.dual_mlp import DualMLP, SwiGLUMLP

# Qwen2.5-1.5B dimensions
HIDDEN_SIZE = 1536
INTERMEDIATE_SIZE = 8960

# Create a standalone DualMLP (as if from scratch)
dual = DualMLP(
    hidden_size=HIDDEN_SIZE,
    intermediate_size=INTERMEDIATE_SIZE,
    alpha_initial=0.95,
    tfidf_threshold=0.3,
)

# Count parameters
frozen_params = sum(p.numel() for p in dual.frozen_mlp.parameters())
trainable_params = sum(p.numel() for p in dual.trainable_mlp.parameters())
total_params = frozen_params + trainable_params

print(f"Frozen MLP parameters:    {frozen_params:>12,}")
print(f"Trainable MLP parameters: {trainable_params:>12,}")
print(f"Total DualMLP parameters: {total_params:>12,}")
print(f"Overhead ratio:           {trainable_params / frozen_params:.1%}")
print()

# Verify: trainable params start near zero, so output ~= frozen output
x = torch.randn(1, 10, HIDDEN_SIZE)  # batch=1, seq_len=10
with torch.no_grad():
    output = dual(x)
    frozen_only = dual.frozen_mlp(x)
    diff = (output - frozen_only).abs().mean().item()
print(f"Mean absolute difference from frozen output: {diff:.6f}")
print("(Should be very small since trainable MLP is near-zero initialized)")

---

## 3. Concepts: TF-IDF Gating

### The Problem: General Knowledge Overwriting

Even with a separate trainable MLP, we face a subtlety: not all neurons in the
trainable MLP are equally important. Some neurons encode **general linguistic knowledge**
(grammar, syntax, common word relationships), while others are more **content-specific**.

If we let gradients update all neurons equally, we risk disrupting the general patterns
that the trainable MLP picks up from the frozen MLP's architecture.

### The Solution: TF-IDF for Neurons

We borrow the **TF-IDF** concept from information retrieval and apply it to *neurons*
instead of words:

| IR Concept | Neuron Analogue |
|------------|----------------|
| **Term** | A neuron's activation |
| **Document** | A text passage |
| **Term Frequency (TF)** | How strongly a neuron activates on the *current* document |
| **Document Frequency (DF)** | How often a neuron activates across a *calibration corpus* |
| **Inverse Document Frequency (IDF)** | log(N / (1 + DF)) -- rarity of the neuron |

### How Calibration Works

1. We run a **calibration corpus** (e.g., WikiText samples) through the model
2. For each calibration text, we record which neurons activate above average
3. We compute the **document frequency**: how many calibration texts activated each neuron
4. **IDF** = log(num_docs / (1 + doc_freq)) -- neurons that activate on everything get low IDF

### How Masking Works at Learning Time

When learning a new document:
1. Compute **TF**: the neuron's activation strength on this document (normalized)
2. Compute **TF-IDF** = TF * IDF
3. Normalize and threshold: neurons with TF-IDF >= 0.3 get mask=1, others get mask=0

**High TF-IDF** = neuron is active on this document AND rare across the corpus
  --> This neuron is document-specific --> gradient passes through

**Low TF-IDF** = neuron is common across the corpus (general-purpose)
  --> Protecting this neuron --> gradient masked to zero

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from continual_learning.model.tfidf_gate import TFIDFGate

# Create a TF-IDF gate
gate = TFIDFGate(hidden_size=256, threshold=0.3)

# Simulate calibration with synthetic activations
# Imagine 20 "documents" from WikiText
torch.manual_seed(42)
calibration_activations = []
for i in range(20):
    # Some neurons activate frequently (general), others rarely (specialized)
    act = torch.randn(1, 32, 256).abs()
    # Make first 50 neurons very active on every document (general-purpose)
    act[:, :, :50] *= 3.0
    # Make neurons 200-220 active on only a few documents (specialized)
    if i < 3:
        act[:, :, 200:220] *= 5.0
    else:
        act[:, :, 200:220] *= 0.1
    calibration_activations.append(act)

# Run calibration
gate.calibrate(calibration_activations)

print(f"Calibrated on {gate._num_docs} documents")
print(f"IDF scores shape: {gate.idf_scores.shape}")
print(f"IDF range: [{gate.idf_scores.min():.3f}, {gate.idf_scores.max():.3f}]")
print()

# Now simulate a new document that activates the specialized neurons
new_doc_activation = torch.randn(1, 32, 256).abs()
new_doc_activation[:, :, 200:220] *= 5.0  # This document uses the specialized neurons

mask = gate.compute_mask(new_doc_activation)
num_active = mask.sum().item()
print(f"Neurons allowed to update: {int(num_active)} / {mask.shape[0]}")
print(f"Masking ratio: {1 - num_active/mask.shape[0]:.1%} of neurons blocked")

In [ ]:
# Visualize IDF scores and the resulting mask
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# IDF scores
axes[0].bar(range(256), gate.idf_scores.numpy(), width=1.0, color="steelblue", alpha=0.7)
axes[0].set_xlabel("Neuron Index")
axes[0].set_ylabel("IDF Score")
axes[0].set_title("IDF Scores (higher = rarer, more specialized)")
axes[0].axhline(y=gate.idf_scores.mean().item(), color="red", linestyle="--", label="Mean IDF")
axes[0].legend()

# TF-IDF for the new document
tf = new_doc_activation.abs().mean(dim=(0, 1))
tf = tf / tf.max()
tfidf = tf * gate.idf_scores
tfidf_norm = tfidf / tfidf.max()

axes[1].bar(range(256), tfidf_norm.numpy(), width=1.0, color="darkorange", alpha=0.7)
axes[1].axhline(y=0.3, color="red", linestyle="--", label=f"Threshold = {gate.threshold}")
axes[1].set_xlabel("Neuron Index")
axes[1].set_ylabel("Normalized TF-IDF")
axes[1].set_title("TF-IDF Scores for New Document")
axes[1].legend()

# Binary mask
colors = ["#2ecc71" if m == 1.0 else "#e74c3c" for m in mask.numpy()]
axes[2].bar(range(256), mask.numpy(), width=1.0, color=colors, alpha=0.7)
axes[2].set_xlabel("Neuron Index")
axes[2].set_ylabel("Mask Value")
axes[2].set_title(f"Gradient Mask (green=pass, red=blocked)")
axes[2].set_yticks([0, 1])
axes[2].set_yticklabels(["Blocked", "Pass"])

plt.tight_layout()
plt.show()

---

## 4. Concepts: Alpha Decay

### The Blending Coefficient

The `alpha` parameter controls how much influence the trainable MLP has on the output:

$$\text{output} = \text{frozen\_out} + (1 - \alpha) \cdot \text{gated\_trainable\_out}$$

- **alpha = 1.0** --> trainable contribution is 0% (fully frozen behavior)
- **alpha = 0.5** --> trainable contribution is 50% (balanced)
- **alpha = 0.0** --> trainable contribution is 100% (fully trainable)

### Why Decay Alpha?

When the trainable MLP is freshly initialized (near-zero weights), its outputs are
essentially noise. Giving it too much influence early on would degrade the model.

As the model learns more documents, the trainable MLP's weights become more meaningful.
We gradually increase its influence by **decaying alpha**:

$$\alpha_{\text{new}} = \max(\alpha_{\min}, \alpha_{\text{current}} - \text{decay\_rate})$$

The default schedule from `configs/default.yaml`:
- `alpha_initial`: 1.0
- `alpha_decay_rate`: 0.01 per mini-batch
- `alpha_min`: 0.3

In [ ]:
# Plot alpha decay over 100 documents
# Each document has roughly 4096 / 32 = 128 mini-batches
# Alpha decays once per mini-batch

alpha_initial = 1.0
decay_rate = 0.01
alpha_min = 0.3
mini_batches_per_doc = 128  # ~4096 tokens / 32 batch_size

alphas_over_docs = []
alpha = alpha_initial

for doc_idx in range(100):
    alphas_over_docs.append(alpha)
    for _ in range(mini_batches_per_doc):
        alpha = max(alpha_min, alpha - decay_rate)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Alpha decay
ax1.plot(range(100), alphas_over_docs, "b-", linewidth=2)
ax1.axhline(y=alpha_min, color="red", linestyle="--", label=f"alpha_min = {alpha_min}")
ax1.axhline(y=alpha_initial, color="green", linestyle="--", label=f"alpha_initial = {alpha_initial}")
ax1.fill_between(range(100), alphas_over_docs, alpha_min, alpha=0.1, color="blue")
ax1.set_xlabel("Document Index")
ax1.set_ylabel("Alpha")
ax1.set_title("Alpha Decay Over 100 Documents")
ax1.legend()
ax1.set_ylim(0, 1.1)
ax1.grid(True, alpha=0.3)

# Trainable contribution (1 - alpha)
trainable_contribution = [1 - a for a in alphas_over_docs]
ax2.plot(range(100), trainable_contribution, "darkorange", linewidth=2)
ax2.fill_between(range(100), trainable_contribution, 0, alpha=0.1, color="orange")
ax2.set_xlabel("Document Index")
ax2.set_ylabel("Trainable MLP Contribution (1 - alpha)")
ax2.set_title("Trainable MLP Influence Over Time")
ax2.set_ylim(0, 1.1)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find the document where alpha bottoms out
bottomed_out = next((i for i, a in enumerate(alphas_over_docs) if a <= alpha_min), None)
print(f"Alpha reaches minimum ({alpha_min}) at document {bottomed_out}")
print(f"After that, trainable MLP contributes {1-alpha_min:.0%} to the output")

---

## 5. Hands-On: Loading and Modifying the Model

Now let's work with the real model. We will:
1. Load Qwen2.5-1.5B from HuggingFace
2. Inject DualMLPs into the last 7 transformer layers (21-28)
3. Inspect the modified architecture

**Note:** This requires a GPU with at least 8GB of VRAM, or sufficient CPU RAM (~6GB).

In [ ]:
from continual_learning.model.modified_qwen import load_modified_model, inject_dual_mlps
from continual_learning.model.dual_mlp import DualMLP

# Configuration (matches configs/default.yaml)
MODEL_NAME = "Qwen/Qwen2.5-1.5B"
START_LAYER = 21
END_LAYER = 28
ALPHA_INITIAL = 1.0
TFIDF_THRESHOLD = 0.3

# Load and modify the model
print(f"Loading {MODEL_NAME}...")
model, tokenizer = load_modified_model(
    model_name=MODEL_NAME,
    device="auto",
    start_layer=START_LAYER,
    end_layer=END_LAYER,
    alpha_initial=ALPHA_INITIAL,
    tfidf_threshold=TFIDF_THRESHOLD,
)
print("Model loaded and modified.")

In [ ]:
# Inspect which layers were modified
num_layers = len(model.model.layers)
print(f"Total transformer layers: {num_layers}")
print(f"Modified layers: {START_LAYER} to {END_LAYER - 1}")
print()

dual_mlps = []
for i, layer in enumerate(model.model.layers):
    mlp = layer.mlp
    is_dual = isinstance(mlp, DualMLP)
    marker = " <-- DualMLP injected" if is_dual else ""
    if is_dual or i >= START_LAYER - 2:  # Show a few layers before the modified range
        print(f"  Layer {i:2d}: {type(mlp).__name__}{marker}")
    if is_dual:
        dual_mlps.append(mlp)

print(f"\nTotal DualMLP modules: {len(dual_mlps)}")

In [ ]:
# Parameter summary: frozen vs trainable
total_frozen = 0
total_trainable = 0

for name, param in model.named_parameters():
    if param.requires_grad:
        total_trainable += param.numel()
    else:
        total_frozen += param.numel()

total_all = total_frozen + total_trainable

print("Parameter Summary")
print("=" * 50)
print(f"Frozen parameters:    {total_frozen:>14,}  ({total_frozen/total_all:.1%})")
print(f"Trainable parameters: {total_trainable:>14,}  ({total_trainable/total_all:.1%})")
print(f"Total parameters:     {total_all:>14,}")
print()
print(f"Trainable size (est): ~{total_trainable * 4 / 1024**2:.0f} MB (float32)")
print(f"Full model size:      ~{total_all * 4 / 1024**2:.0f} MB (float32)")
print()
print("Only the trainable parameters receive gradient updates during TTT.")
print("Checkpoints only save the trainable weights (~50-100 MB instead of the full model).")

---

## 6. Hands-On: Learning a Document

Now we feed a document to the TTT engine. The engine will:
1. Tokenize the document
2. Split it into mini-batches (32 tokens each)
3. For each mini-batch, run forward + backward pass
4. Apply TF-IDF gradient masks
5. Update only trainable MLP parameters
6. Decay alpha after each mini-batch

In [ ]:
# First, calibrate the TF-IDF gates using some general text
from continual_learning.training.calibration import calibrate_gates

# Use a few general-knowledge passages for calibration
calibration_texts = [
    "The Earth orbits the Sun at an average distance of about 93 million miles. "
    "It takes approximately 365.25 days for the Earth to complete one full orbit.",
    
    "Water is a chemical compound with the formula H2O. Each molecule consists of "
    "one oxygen atom covalently bonded to two hydrogen atoms.",
    
    "The Python programming language was created by Guido van Rossum and first "
    "released in 1991. It emphasizes code readability and simplicity.",
    
    "Photosynthesis is the process by which green plants convert sunlight into "
    "chemical energy. Carbon dioxide and water are transformed into glucose and oxygen.",
    
    "The Roman Empire was one of the largest empires in ancient history. At its "
    "peak under Emperor Trajan, it encompassed about 5 million square kilometers.",
]

print("Calibrating TF-IDF gates...")
calibrate_gates(model, tokenizer, calibration_texts, dual_mlps, max_tokens=512)
print(f"Calibration complete. Gates calibrated on {len(calibration_texts)} texts.")

# Show calibration stats for the first DualMLP
gate = dual_mlps[0].gate
print(f"\nGate stats for layer {START_LAYER}:")
print(f"  IDF scores range: [{gate.idf_scores.min():.3f}, {gate.idf_scores.max():.3f}]")
print(f"  Mean IDF: {gate.idf_scores.mean():.3f}")

In [ ]:
from continual_learning.training.ttt_engine import TTTEngine

# Read the sample document
with open("data/sample_document.txt", "r") as f:
    document = f.read()

print(f"Document length: {len(document)} characters")
print(f"Document preview: {document[:200]}...")
print()

# Create the TTT engine
engine = TTTEngine(
    model=model,
    tokenizer=tokenizer,
    dual_mlps=dual_mlps,
    learning_rate=1e-5,
    mini_batch_size=32,
    gradient_steps=1,
    max_tokens=4096,
    alpha_decay_rate=0.01,
    alpha_min=0.3,
)

# Track progress during learning
progress_data = []

def on_batch(batch_idx, loss, tokens_so_far):
    progress_data.append({"batch": batch_idx, "loss": loss, "tokens": tokens_so_far})
    if batch_idx % 5 == 0:
        print(f"  Batch {batch_idx:3d} | Loss: {loss:.4f} | Tokens: {tokens_so_far}")

# Run the TTT inner loop
print("Learning document...")
result = engine.learn(document, callback=on_batch)

print(f"\nLearning complete!")
print(f"  Total tokens processed: {result['tokens_processed']}")
print(f"  Number of mini-batches: {len(result['losses'])}")
print(f"  Initial loss: {result['losses'][0]:.4f}")
print(f"  Final loss:   {result['losses'][-1]:.4f}")
print(f"  Loss reduction: {result['losses'][0] - result['losses'][-1]:.4f}")

In [ ]:
# Plot the loss curve during learning
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

losses = result["losses"]

# Raw loss
ax1.plot(losses, "b-", alpha=0.7, linewidth=1)
ax1.set_xlabel("Mini-batch")
ax1.set_ylabel("Cross-Entropy Loss")
ax1.set_title("Loss During Document Learning")
ax1.grid(True, alpha=0.3)

# Smoothed loss (moving average)
if len(losses) > 5:
    window = min(5, len(losses))
    smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
    ax2.plot(range(len(smoothed)), smoothed, "darkorange", linewidth=2)
    ax2.set_xlabel("Mini-batch")
    ax2.set_ylabel("Smoothed Loss")
    ax2.set_title(f"Smoothed Loss (window={window})")
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, "Too few batches to smooth", ha="center", va="center",
             transform=ax2.transAxes)

plt.tight_layout()
plt.show()

# Print alpha values after learning
print("Alpha values after learning:")
for i, dual in enumerate(dual_mlps):
    print(f"  Layer {START_LAYER + i}: alpha = {dual.alpha:.4f}")

---

## 7. Hands-On: Testing Knowledge

After learning the document, let's test whether the model has internalized the information.
We will:
1. Ask questions about the document's content
2. See if the model can produce relevant completions
3. Check for forgetting on general knowledge

In [ ]:
def generate_answer(model, tokenizer, prompt, max_new_tokens=100):
    """Generate text from a prompt using the model."""
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Greedy decoding for reproducibility
            temperature=1.0,
        )
    
    # Decode only the generated tokens (not the prompt)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

# Questions about the learned document
document_questions = [
    "The primary active compounds in valerian root are",
    "The recommended dosage for valerian root extract is",
    "A 2006 meta-analysis of valerian examined",
    "Valerenic acid works by inhibiting the breakdown of",
]

print("=" * 70)
print("KNOWLEDGE TEST: Questions About the Learned Document")
print("=" * 70)

for q in document_questions:
    answer = generate_answer(model, tokenizer, q, max_new_tokens=80)
    print(f"\nPrompt: {q}")
    print(f"Model:  {answer[:200]}")
    print("-" * 70)

In [ ]:
# Forgetting test: ask general knowledge questions
# These should still work after learning the valerian document

holdout_questions = [
    "The capital of France is",
    "Water boils at a temperature of",
    "The speed of light is approximately",
]

print("=" * 70)
print("FORGETTING TEST: General Knowledge (Should Still Work)")
print("=" * 70)

for q in holdout_questions:
    answer = generate_answer(model, tokenizer, q, max_new_tokens=50)
    print(f"\nPrompt: {q}")
    print(f"Model:  {answer[:200]}")
    print("-" * 70)

print("\nIf the general knowledge answers are still correct, the DualMLP + TF-IDF")
print("gating is successfully preventing catastrophic forgetting.")

---

## 8. Hands-On: Checkpointing

One of the key advantages of the DualMLP approach is **efficient checkpointing**.
Since we only need to save the trainable MLP weights and TF-IDF gate statistics,
checkpoints are much smaller than the full model:

- Full Qwen2.5-1.5B model: ~3 GB
- DualMLP checkpoint: ~50-100 MB

This means you can save a "knowledge snapshot" for each learned document and
load them on demand.

In [ ]:
from continual_learning.checkpointing.manager import CheckpointManager
import tempfile

# Use a temporary directory for this tutorial's checkpoints
checkpoint_dir = tempfile.mkdtemp(prefix="ttt_tutorial_")
manager = CheckpointManager(checkpoint_dir=checkpoint_dir)

# Save the current learned state
metadata = {
    "document": "sample_document.txt",
    "tokens_processed": result["tokens_processed"],
    "final_loss": result["losses"][-1] if result["losses"] else None,
    "num_batches": len(result["losses"]),
    "model_name": MODEL_NAME,
    "layers": f"{START_LAYER}-{END_LAYER}",
}

save_path = manager.save("valerian_knowledge", dual_mlps, metadata)
print(f"Checkpoint saved to: {save_path}")

# Show checkpoint contents
import os
print(f"\nCheckpoint contents:")
total_size = 0
for fname in sorted(os.listdir(save_path)):
    fpath = os.path.join(save_path, fname)
    size = os.path.getsize(fpath)
    total_size += size
    print(f"  {fname}: {size / 1024:.1f} KB")
print(f"  Total: {total_size / 1024 / 1024:.1f} MB")

In [ ]:
# Verify: reset trainable weights, then reload checkpoint

# Step 1: Record current alpha values and a weight sample
alpha_before = [d.alpha for d in dual_mlps]
weight_sample_before = dual_mlps[0].trainable_mlp.gate_proj.weight.data[0, :5].clone()
print("Before reset:")
print(f"  Alpha[0] = {alpha_before[0]:.4f}")
print(f"  Weight sample: {weight_sample_before}")

# Step 2: Reset trainable MLPs to random (simulating fresh start)
for dual in dual_mlps:
    for param in dual.trainable_mlp.parameters():
        nn.init.normal_(param, mean=0.0, std=0.001)
    dual.alpha = ALPHA_INITIAL  # Reset alpha

weight_sample_reset = dual_mlps[0].trainable_mlp.gate_proj.weight.data[0, :5].clone()
print("\nAfter reset:")
print(f"  Alpha[0] = {dual_mlps[0].alpha:.4f}")
print(f"  Weight sample: {weight_sample_reset}")

# Step 3: Load the checkpoint
loaded_meta = manager.load("valerian_knowledge", dual_mlps)

weight_sample_loaded = dual_mlps[0].trainable_mlp.gate_proj.weight.data[0, :5].clone()
print("\nAfter loading checkpoint:")
print(f"  Alpha[0] = {dual_mlps[0].alpha:.4f}")
print(f"  Weight sample: {weight_sample_loaded}")
print(f"  Metadata: {loaded_meta}")

# Verify weights match
match = torch.allclose(weight_sample_before, weight_sample_loaded)
print(f"\nWeights restored correctly: {match}")

# Clean up temp directory
import shutil
shutil.rmtree(checkpoint_dir)
print(f"Cleaned up temporary checkpoint directory.")

---

## 9. Deep Dive: The Math

Let's formalize the TTT-E2E algorithm.

### Notation

- $\theta_f$ -- frozen MLP parameters (never updated)
- $\theta_t$ -- trainable MLP parameters (updated during TTT)
- $\alpha$ -- blending coefficient
- $M$ -- TF-IDF gradient mask (binary, per-neuron)
- $D = \{x_1, x_2, \ldots, x_T\}$ -- document tokens

### Forward Pass

For each DualMLP layer $l$:

$$h_l = f_{\theta_f}(x) + (1 - \alpha) \cdot M \odot f_{\theta_t}(x)$$

where $f_{\theta}(x)$ is the SwiGLU MLP:

$$f_{\theta}(x) = W_{\text{down}} \cdot (\sigma(W_{\text{gate}} \cdot x) \odot W_{\text{up}} \cdot x)$$

and $\sigma$ is the SiLU activation: $\sigma(z) = z \cdot \text{sigmoid}(z)$.

### Loss Function

We use the standard **causal language modeling** loss (next-token prediction):

$$\mathcal{L} = -\frac{1}{T-1} \sum_{i=1}^{T-1} \log P(x_{i+1} | x_{\leq i}; \theta_f, \theta_t)$$

This is cross-entropy between the model's predicted distribution and the actual next token.
The loss is computed *only on the document's tokens* -- we are teaching the model to predict
the document itself.

### Gradient Masking

After computing $\nabla_{\theta_t} \mathcal{L}$, we apply the TF-IDF mask:

$$\nabla_{\theta_t} \mathcal{L} \leftarrow \nabla_{\theta_t} \mathcal{L} \odot M_{\text{TF-IDF}}$$

where $M_{\text{TF-IDF}}$ is a binary mask computed as:

$$\text{TF}_j = \frac{|a_j|}{\max_k |a_k|}$$

$$\text{IDF}_j = \log \frac{N}{1 + \text{DF}_j}$$

$$M_j = \mathbb{1}\left[\frac{\text{TF}_j \cdot \text{IDF}_j}{\max_k (\text{TF}_k \cdot \text{IDF}_k)} \geq \tau\right]$$

where $\tau = 0.3$ is the threshold and $N$ is the number of calibration documents.

### Update Rule

We use Adam optimizer with learning rate $\eta = 10^{-5}$:

$$\theta_t \leftarrow \text{Adam}(\theta_t, \nabla_{\theta_t} \mathcal{L} \odot M_{\text{TF-IDF}}, \eta)$$

### Alpha Decay

After each mini-batch:

$$\alpha \leftarrow \max(\alpha_{\min}, \alpha - \delta)$$

where $\delta = 0.01$ is the decay rate and $\alpha_{\min} = 0.3$.

### Key Properties

1. **No forgetting of frozen knowledge**: $\theta_f$ is never modified
2. **Selective updates**: TF-IDF mask ensures only document-specific neurons update
3. **Gradual integration**: Alpha decay smoothly increases trainable MLP influence
4. **Efficient storage**: Only $\theta_t$, gate statistics, and alpha values need checkpointing

---

## 10. Exercises

Try these experiments to build intuition about how TTT-E2E behaves under different settings.

### Exercise 1: Alpha Sensitivity

How does the initial alpha value affect learning? Try three settings:
- `alpha_initial = 0.5` (aggressive -- trainable MLP has 50% influence from the start)
- `alpha_initial = 0.8` (moderate)
- `alpha_initial = 1.0` (conservative -- default, trainable MLP starts with 0% influence)

For each, reload the model fresh, learn the same document, and compare:
- Final loss
- Quality of answers to document questions
- Whether general knowledge is preserved

In [ ]:
# Exercise 1: Alpha Sensitivity
#
# Uncomment and run this cell to compare alpha settings.
# WARNING: This reloads the model 3 times, which may take several minutes.

# from continual_learning.model.modified_qwen import load_modified_model
# from continual_learning.training.ttt_engine import TTTEngine
# from continual_learning.training.calibration import calibrate_gates
#
# alpha_values = [0.5, 0.8, 1.0]
# results_by_alpha = {}
#
# with open("data/sample_document.txt", "r") as f:
#     document = f.read()
#
# for alpha_init in alpha_values:
#     print(f"\n{'='*50}")
#     print(f"Testing alpha_initial = {alpha_init}")
#     print(f"{'='*50}")
#
#     model_ex, tok_ex = load_modified_model(
#         model_name="Qwen/Qwen2.5-1.5B",
#         device="auto",
#         start_layer=21, end_layer=28,
#         alpha_initial=alpha_init,
#     )
#
#     duals = [layer.mlp for layer in model_ex.model.layers
#              if isinstance(layer.mlp, DualMLP)]
#
#     calibrate_gates(model_ex, tok_ex, calibration_texts, duals)
#
#     engine_ex = TTTEngine(
#         model=model_ex, tokenizer=tok_ex, dual_mlps=duals,
#         learning_rate=1e-5, mini_batch_size=32,
#         alpha_decay_rate=0.01, alpha_min=0.3,
#     )
#
#     res = engine_ex.learn(document)
#     results_by_alpha[alpha_init] = res
#     print(f"  Final loss: {res['losses'][-1]:.4f}")
#     print(f"  Alpha after: {duals[0].alpha:.4f}")
#
#     # Clean up GPU memory
#     del model_ex, tok_ex, engine_ex, duals
#     torch.cuda.empty_cache() if torch.cuda.is_available() else None
#
# # Plot comparison
# fig, ax = plt.subplots(figsize=(10, 5))
# for alpha_init, res in results_by_alpha.items():
#     ax.plot(res["losses"], label=f"alpha_init={alpha_init}")
# ax.set_xlabel("Mini-batch")
# ax.set_ylabel("Loss")
# ax.set_title("Effect of Initial Alpha on Learning")
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.show()

print("Exercise 1: Uncomment the code above and run to compare alpha settings.")
print("Expected observations:")
print("  - Lower alpha_initial --> faster loss decrease (more aggressive learning)")
print("  - Lower alpha_initial --> higher risk of degraded general knowledge")
print("  - alpha_initial=1.0 is safest but slowest to internalize new info")

### Exercise 2: Layer Range

Which transformer layers are best for knowledge injection? Try:
- Layers 10-17 (middle layers)
- Layers 21-28 (later layers -- default)

Research suggests that later layers store more factual knowledge, while earlier
layers handle syntax and grammar. But does this hold for TTT-E2E?

In [ ]:
# Exercise 2: Layer Range Comparison
#
# Uncomment and run this cell to compare layer ranges.
# WARNING: This reloads the model 2 times.

# layer_configs = [
#     (10, 17, "Layers 10-16 (middle)"),
#     (21, 28, "Layers 21-27 (later, default)"),
# ]
# results_by_layers = {}
#
# for start, end, label in layer_configs:
#     print(f"\n{'='*50}")
#     print(f"Testing {label}")
#     print(f"{'='*50}")
#
#     model_ex, tok_ex = load_modified_model(
#         model_name="Qwen/Qwen2.5-1.5B",
#         device="auto",
#         start_layer=start, end_layer=end,
#         alpha_initial=1.0,
#     )
#
#     duals = [layer.mlp for layer in model_ex.model.layers
#              if isinstance(layer.mlp, DualMLP)]
#
#     calibrate_gates(model_ex, tok_ex, calibration_texts, duals)
#
#     engine_ex = TTTEngine(
#         model=model_ex, tokenizer=tok_ex, dual_mlps=duals,
#         learning_rate=1e-5, mini_batch_size=32,
#         alpha_decay_rate=0.01, alpha_min=0.3,
#     )
#
#     res = engine_ex.learn(document)
#     results_by_layers[label] = res
#     print(f"  Final loss: {res['losses'][-1]:.4f}")
#
#     del model_ex, tok_ex, engine_ex, duals
#     torch.cuda.empty_cache() if torch.cuda.is_available() else None
#
# fig, ax = plt.subplots(figsize=(10, 5))
# for label, res in results_by_layers.items():
#     ax.plot(res["losses"], label=label)
# ax.set_xlabel("Mini-batch")
# ax.set_ylabel("Loss")
# ax.set_title("Effect of Layer Range on Learning")
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.show()

print("Exercise 2: Uncomment the code above and run to compare layer ranges.")
print("Expected observations:")
print("  - Later layers (21-27) typically learn factual content faster")
print("  - Middle layers (10-16) may learn syntactic patterns better")
print("  - Later layers are the default because TTT-E2E targets factual knowledge")

### Exercise 3: Sequential Learning and Forgetting

The ultimate test of continual learning: learn 5 documents sequentially and measure
whether knowledge of earlier documents is retained.

You will need to:
1. Create 5 short documents on different topics
2. Learn each document sequentially using the same TTT engine
3. After each document, test retention of *all* previously learned documents
4. Plot forgetting over time

In [ ]:
# Exercise 3: Sequential Learning and Forgetting
#
# Uncomment and run to explore catastrophic forgetting behavior.

# documents = [
#     "The Atacama Desert in Chile is the driest non-polar desert on Earth. "
#     "Some weather stations in the Atacama have never recorded rainfall. "
#     "Despite its aridity, the Atacama is home to specialized microorganisms "
#     "that extract moisture from fog.",
#
#     "CRISPR-Cas9 is a gene editing technology discovered in 2012 by Jennifer "
#     "Doudna and Emmanuelle Charpentier. It allows precise cuts to DNA sequences "
#     "and has revolutionized genetic research and potential disease treatment.",
#
#     "The Great Barrier Reef extends over 2,300 kilometers along the coast of "
#     "Queensland, Australia. It is the largest coral reef system on Earth and "
#     "is visible from space. It contains over 1,500 species of fish.",
#
#     "Quantum entanglement is a phenomenon where two particles become connected "
#     "such that measuring one instantly affects the other, regardless of distance. "
#     "Einstein famously called it spooky action at a distance.",
#
#     "The Voyager 1 spacecraft, launched in 1977, is the most distant human-made "
#     "object from Earth. In 2012, it crossed the heliopause and entered interstellar "
#     "space, more than 14 billion miles from the Sun.",
# ]
#
# test_prompts = [
#     "The driest non-polar desert on Earth is",
#     "CRISPR-Cas9 was discovered in",
#     "The Great Barrier Reef extends over",
#     "Quantum entanglement was called by Einstein",
#     "Voyager 1 entered interstellar space in",
# ]
#
# # Learn documents sequentially and track perplexity on each
# forgetting_matrix = []  # forgetting_matrix[i][j] = loss on doc j after learning doc i
#
# model_ex, tok_ex = load_modified_model(
#     model_name="Qwen/Qwen2.5-1.5B", device="auto",
#     start_layer=21, end_layer=28, alpha_initial=1.0,
# )
# duals = [l.mlp for l in model_ex.model.layers if isinstance(l.mlp, DualMLP)]
# calibrate_gates(model_ex, tok_ex, calibration_texts, duals)
# engine_ex = TTTEngine(
#     model=model_ex, tokenizer=tok_ex, dual_mlps=duals,
#     learning_rate=1e-5, mini_batch_size=32,
#     alpha_decay_rate=0.005, alpha_min=0.3,
# )
#
# for doc_idx, doc in enumerate(documents):
#     print(f"\nLearning document {doc_idx + 1}: {doc[:50]}...")
#     engine_ex.learn(doc)
#
#     # Test all prompts after learning this document
#     row = []
#     for prompt in test_prompts:
#         answer = generate_answer(model_ex, tok_ex, prompt, max_new_tokens=30)
#         row.append(answer[:80])
#     forgetting_matrix.append(row)
#
# # Print forgetting matrix
# print("\n" + "="*70)
# print("FORGETTING MATRIX")
# print("Rows = after learning doc N, Columns = test prompt for doc M")
# print("="*70)
# for i, row in enumerate(forgetting_matrix):
#     print(f"\nAfter learning doc {i+1}:")
#     for j, answer in enumerate(row):
#         marker = " <-- just learned" if j == i else ""
#         print(f"  Doc {j+1} prompt: {answer}{marker}")
#
# del model_ex, tok_ex, engine_ex, duals
# torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("Exercise 3: Uncomment the code above and run to explore forgetting.")
print("Expected observations:")
print("  - Early documents may show slight degradation as more are learned")
print("  - TF-IDF gating should reduce forgetting compared to naive fine-tuning")
print("  - Alpha decay rate affects the tradeoff: slower decay = less forgetting")

---

## Summary

In this tutorial, we explored the TTT-E2E strategy for continual learning:

| Component | Purpose |
|-----------|--------|
| **DualMLP** | Adds a trainable MLP alongside each frozen MLP in layers 21-27 |
| **TF-IDF Gating** | Masks gradients to protect general-purpose neurons |
| **Alpha Decay** | Gradually increases trainable MLP influence over time |
| **TTT Engine** | Mini-batch gradient descent on documents at inference time |
| **Checkpointing** | Saves only trainable weights (~50-100 MB vs ~3 GB full model) |

### Key Takeaways

1. **TTT-E2E lets models learn at inference time** without catastrophic forgetting
2. **The DualMLP design is a residual architecture** -- frozen output is always preserved
3. **TF-IDF gating is the key anti-forgetting mechanism** -- it identifies which neurons are document-specific
4. **Alpha decay provides a smooth transition** from frozen to balanced behavior
5. **Efficient checkpointing** makes it practical to save per-document knowledge snapshots

### Next Steps

- Explore the **JitRL** engines for retrieval-augmented alternatives (see `src/continual_learning/jitrl/`)
- Try **Doc-to-LoRA** for hypernetwork-generated adaptations (see `src/continual_learning/doc2lora/`)
- Use the **interactive CLI** (`continual-learning` command) for a guided experience